# ML Kernel Roofline Analysis

Load benchmark CSV output and compare latency, throughput, estimated bandwidth, numerical error, and Roofline position. The bandwidth and arithmetic-intensity values use logical tensor traffic, not hardware counters.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

csv_path = Path('../results/measurements.csv')
df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Latency and throughput versus problem size. Matmul size is N for NxN @ NxN;
# elementwise size is the number of elements.
for operation, group in df.groupby('operation'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for (device, precision), series in group.groupby(['device', 'precision']):
        series = series.sort_values('size')
        label = f'{device}, {precision}'
        axes[0].plot(series['size'], series['median_ms'], marker='o', label=label)
        axes[1].plot(series['size'], series['gflops'], marker='o', label=label)
    for ax, ylabel, title in [(axes[0], 'Median time (ms)', 'Latency'),
                              (axes[1], 'GFLOP/s', 'Throughput')]:
        ax.set_xscale('log', base=2)
        ax.set_yscale('log')
        ax.set_xlabel('Problem size')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{operation}: {title}')
        ax.grid(True, which='both', alpha=.25)
        ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Roofline with the published M2 Pro memory-bandwidth figure as a reference.
# Set this to a sourced compute peak for the relevant precision if one is established.
MEMORY_BANDWIDTH_GB_S = 200.0
COMPUTE_PEAK_GFLOPS = None  # Optional precision-specific compute roof

fig, ax = plt.subplots(figsize=(8, 5))
for (operation, device, precision), group in df.groupby(['operation', 'device', 'precision']):
    ax.scatter(group['arithmetic_intensity'], group['gflops'],
               label=f'{operation}, {device}, {precision}', alpha=.8)
x_min = max(df['arithmetic_intensity'].min() / 2, 1e-3)
x_max = df['arithmetic_intensity'].max() * 2
xs = pd.Series([x_min, x_max])
ax.plot(xs, MEMORY_BANDWIDTH_GB_S * xs, '--', color='black',
        label='200 GB/s bandwidth reference')
if COMPUTE_PEAK_GFLOPS is not None:
    ax.axhline(COMPUTE_PEAK_GFLOPS, linestyle=':', color='gray',
               label='compute peak')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Arithmetic intensity (operations/byte)')
ax.set_ylabel('Throughput (GFLOP/s)')
ax.set_title('Roofline reference and measured kernels')
ax.grid(True, which='both', alpha=.25)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Error relative to the FP32 CPU reference.
for operation, group in df.groupby('operation'):
    fig, ax = plt.subplots(figsize=(7, 4))
    for (device, precision), series in group.groupby(['device', 'precision']):
        series = series.sort_values('size')
        ax.plot(series['size'], series['max_abs_error'], marker='o',
                label=f'{device}, {precision}')
    ax.set_xscale('log', base=2)
    ax.set_yscale('symlog', linthresh=1e-8)
    ax.set_xlabel('Problem size')
    ax.set_ylabel('Maximum absolute error')
    ax.set_title(f'{operation}: numerical error')
    ax.grid(True, which='both', alpha=.25)
    ax.legend()
    plt.tight_layout()
    plt.show()